In [1]:
import warnings
warnings.filterwarnings(action='ignore')
import requests
import folium

아래의 명령을 실행해서 pandas를 설치한다.  
`!pip install pandas`

파이썬 데이터 처리 라이브러리인 pandas를 사용하기 위해서 import 한다.

In [2]:
import pandas as pd

파이썬의 딕셔너리 타입의 데이터를 판다스 데이터프레임으로 변환하기 위해 json_normalize를 import 한다.

In [3]:
from pandas import json_normalize

## 전국 또는 특정 지역의 스타벅스 매장 위치 정보를 가져와서 지도 시각화 한다.

requests 모듈로 스타벅스 매장 정보를 가져와서 딕셔너리 타입으로 변환한다.

In [4]:
targetSite = 'https://www.starbucks.co.kr/store/getStore.do?r=P4K5ZAPLE6'
response = requests.post(url=targetSite, data={
    'ins_lat': 37.365,
    'ins_lng': 127.0639,
    'p_sido_cd': '',
    'p_gugun_cd': '0833',
    'in_biz_cd': '',
    'iend': 2300,
    'set_date': ''
})
storeList = response.json()

for index, store in enumerate(storeList['list'], start=1):
    print(f'{index:4d}. {store["s_name"]}: {store["doro_address"]}({store["lat"]}, {store["lot"]})')

   1. 판교제2테크노밸리: 경기도 성남시 수정구 창업로 54 (시흥동)(37.41108510155048, 127.09463155093412)
   2. 위례중앙광장: 경기도 성남시 수정구 위례광장로 300 (창곡동) 130호(37.473711672971596, 127.1424768773468)
   3. 판교놀유니버스: 경기도 성남시 수정구 금토로 70 (금토동)(37.405652966500696, 127.08746383906066)
   4. 남위례역: 경기도 성남시 수정구 위례광장로 9-9 (창곡동)(37.4646, 127.14142)
   5. 성남모란삼성: 경기도 성남시 수정구 산성대로 81 (수진동)(37.43359712142669, 127.12801723528008)
   6. 판교위든타워: 경기도 성남시 수정구 금토로80번길 56 (금토동)(37.406853849128, 127.090099780526)
   7. 판교아이스퀘어: 경기도 성남시 수정구 창업로 18(시흥동)(37.41291102051743, 127.09848855739167)
   8. 성남태평: 경기도 성남시 수정구 수정로 149 (태평동) 1~2층(37.442561275669, 127.136588869764)
   9. 위례창곡: 경기도 성남시 수정구 위례동로 135 (창곡동)(37.472159, 127.150206)
  10. 수진역: 경기도 성남시 수정구 산성대로 223 (신흥동) 1층(37.4389903651714, 127.14224989005)
  11. 가천대학교: 경기도 성남시 수정구 성남대로 1342 (복정동)(37.44941, 127.1272)
  12. 성남이마트: 경기도 성남시 수정구 수정로 201 (태평1동) 신세계쉐덴(37.4442471, 127.14145710000002)
  13. 성남위례: 경기도 성남시 수정구 위례광장로 104 (창곡동)(37.472839, 127.141062)
  14. 신흥역: 경기도 성남시 수정구 산성대로 267 (신흥동)(37

json_normalize() 함수를 이용해서 딕셔너리를 판다스 데이터프레임으로 변환한다.

json_normalize(데이터프레임으로 변환하려는 딕셔너리, 데이터프레임으로 변환하려는 딕셔너리의 key)

In [5]:
starbucks_df = json_normalize(storeList, 'list')
# print(type(starbucks_df))
# print(len(starbucks_df))
# print(starbucks_df.columns)
starbucks_df

,seq,sido_cd,sido_nm,gugun_cd,gugun_nm,code_order,view_yn,store_num,sido,gugun,...,t30,t36,t27,t29,t43,t48,z9999,t64,t66,p02
0,0,None,None,None,None,None,None,None,None,None,...,0,0,0,0,0,0,0,0,0,0
1,0,None,None,None,None,None,None,None,None,None,...,0,0,0,0,0,0,0,0,0,0
2,0,None,None,None,None,None,None,None,None,None,...,0,0,0,0,0,0,0,0,0,0
3,0,None,None,None,None,None,None,None,None,None,...,0,0,0,0,0,0,0,0,0,0
4,0,None,None,None,None,None,None,None,None,None,...,0,0,0,0,0,0,0,0,0,0
5,0,None,None,None,None,None,None,None,None,None,...,0,0,0,0,0,0,0,0,0,0
6,0,None,None,None,None,None,None,None,None,None,...,0,0,0,0,0,0,0,0,0,0
7,0,None,None,None,None,None,None,None,None,None,...,0,0,0,0,0,0,0,0,0,0
8,0,None,None,None,None,None,None,None,None,None,...,0,0,0,0,0,0,0,0,0,0
9,0,None,None,None,None,None,None,None,None,None,...,0,0,0,0,0,0,0,0,0,0


딕셔너리를 데이터프레임으로 변환하니 경기도 성남시 수정구에 있는 스타벅스 지점은 14개고 지점당 137개의 컬럼으로 정보를 표시하고 있다.  
작업에 필요한 컬럼 몇 가지를 선택해서 지도 시각화에 사용할 데이터프레임을 만든다.

지점이름: s_name  
시도코드: sido_code  
시도이름: sido_name  
구군코드: gugun_code  
구군이름: gugun_name  
도로명주소: doro_address  
위도: lat  
경도: lot

In [6]:
starbucks_df = starbucks_df[
    ['s_name', 'sido_code', 'sido_name', 'gugun_code', 'gugun_name', 'doro_address', 'lat', 'lot']
]
starbucks_df

,s_name,sido_code,sido_name,gugun_code,gugun_name,doro_address,lat,lot
0,판교제2테크노밸리,08,경기,0833,성남시 수정구,경기도 성남시 수정구 창업로 54 (시흥동),37.41108510155048,127.09463155093412
1,위례중앙광장,08,경기,0833,성남시 수정구,경기도 성남시 수정구 위례광장로 300 (창곡동) 130호,37.473711672971596,127.1424768773468
2,판교놀유니버스,08,경기,0833,성남시 수정구,경기도 성남시 수정구 금토로 70 (금토동),37.405652966500696,127.08746383906066
3,남위례역,08,경기,0833,성남시 수정구,경기도 성남시 수정구 위례광장로 9-9 (창곡동),37.4646,127.14142
4,성남모란삼성,08,경기,0833,성남시 수정구,경기도 성남시 수정구 산성대로 81 (수진동),37.43359712142669,127.12801723528008
5,판교위든타워,08,경기,0833,성남시 수정구,경기도 성남시 수정구 금토로80번길 56 (금토동),37.406853849128,127.090099780526
6,판교아이스퀘어,08,경기,0833,성남시 수정구,경기도 성남시 수정구 창업로 18(시흥동),37.41291102051743,127.09848855739167
7,성남태평,08,경기,0833,성남시 수정구,경기도 성남시 수정구 수정로 149 (태평동) 1~2층,37.442561275669,127.136588869764
8,위례창곡,08,경기,0833,성남시 수정구,경기도 성남시 수정구 위례동로 135 (창곡동),37.472159,127.150206
9,수진역,08,경기,0833,성남시 수정구,경기도 성남시 수정구 산성대로 223 (신흥동) 1층,37.4389903651714,127.14224989005


dtypes 속성이나 info() 메소드로 각 컬럼의 자료형을 확인한다.  
    판다스 2.x 까지는 문자열 데이터를 `str`이 아니라 `object`로 나타낸다.

In [7]:
starbucks_df.dtypes

s_name          str
sido_code       str
sido_name       str
gugun_code      str
gugun_name      str
doro_address    str
lat             str
lot             str
dtype: object

In [8]:
starbucks_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   s_name        14 non-null     str  
 1   sido_code     14 non-null     str  
 2   sido_name     14 non-null     str  
 3   gugun_code    14 non-null     str  
 4   gugun_name    14 non-null     str  
 5   doro_address  14 non-null     str  
 6   lat           14 non-null     str  
 7   lot           14 non-null     str  
dtypes: str(8)
memory usage: 1.0 KB


astype() 메소드로 위도, 경도 데이터의 타입을 str에서 float로 변경한다.

In [9]:
starbucks_df['lat'] = starbucks_df['lat'].astype(float)
starbucks_df['lot'] = starbucks_df['lot'].astype(float)
starbucks_df.dtypes

s_name              str
sido_code           str
sido_name           str
gugun_code          str
gugun_name          str
doro_address        str
lat             float64
lot             float64
dtype: object

## 경기도 성남시 수정구의 스타벅스 매장 위치 시각화

iterrows() 메소드는 데이터프레임에 저장된 데이터의 인덱스와 데이터를 묶은 튜플 형태를 리턴한다.

In [15]:
for _, star in starbucks_df.iterrows():
    print(star)

s_name                         판교제2테크노밸리
sido_code                             08
sido_name                             경기
gugun_code                          0833
gugun_name                       성남시 수정구
doro_address    경기도 성남시 수정구 창업로 54 (시흥동)
lat                            37.411085
lot                           127.094632
Name: 0, dtype: object
s_name                                    위례중앙광장
sido_code                                     08
sido_name                                     경기
gugun_code                                  0833
gugun_name                               성남시 수정구
doro_address    경기도 성남시 수정구 위례광장로 300 (창곡동) 130호
lat                                    37.473712
lot                                   127.142477
Name: 1, dtype: object
s_name                           판교놀유니버스
sido_code                             08
sido_name                             경기
gugun_code                          0833
gugun_name                       성남시 수정구
doro_address    경기도 성남시 수정구 금

folium를 사용해서 표시할 마커가 여러개일 경우 지도의 중심은 마커를 표시할 모든 데이터의 위도와 경도의 평균을 중심으로 한다.

In [20]:
foliumMap = folium.Map(location=[starbucks_df['lat'].mean(), starbucks_df['lot'].mean()], zoom_start=13)

for _, star in starbucks_df.iterrows():
    popup = folium.Popup(f'{star["s_name"]} - {star["doro_address"]}', max_width=300)
    folium.Marker(location=[star['lat'], star['lot']], popup=popup).add_to(foliumMap)

foliumMap

## 서울시 강남구

In [21]:
targetSite = 'https://www.starbucks.co.kr/store/getStore.do?r=P4K5ZAPLE6'
response = requests.post(url=targetSite, data={
    'ins_lat': 37.365,
    'ins_lng': 127.0639,
    'p_sido_cd': '',
    'p_gugun_cd': '0101',
    'in_biz_cd': '',
    'iend': 2300,
    'set_date': ''
})
storeList = response.json()

starbucks_df = json_normalize(storeList, 'list')
starbucks_df = starbucks_df[['s_name', 'sido_code', 'sido_name', 'gugun_code', 'gugun_name', 'doro_address', 'lat', 'lot']]
starbucks_df['lat'] = starbucks_df['lat'].astype(float)
starbucks_df['lot'] = starbucks_df['lot'].astype(float)

foliumMap = folium.Map(location=[starbucks_df['lat'].mean(), starbucks_df['lot'].mean()], zoom_start=13)
for _, star in starbucks_df.iterrows():
    popup = folium.Popup(f'{star["s_name"]} - {star["doro_address"]}', max_width=300)
    folium.Marker(location=[star['lat'], star['lot']], popup=popup).add_to(foliumMap)
foliumMap

## 서울시 전체

In [25]:
targetSite = 'https://www.starbucks.co.kr/store/getStore.do?r=P4K5ZAPLE6'
response = requests.post(url=targetSite, data={
    'ins_lat': 37.365,
    'ins_lng': 127.0639,
    'p_sido_cd': '01',
    'p_gugun_cd': '',
    'in_biz_cd': '',
    'iend': 2300,
    'set_date': ''
})
storeList = response.json()

starbucks_df = json_normalize(storeList, 'list')
starbucks_df = starbucks_df[['s_name', 'sido_code', 'sido_name', 'gugun_code', 'gugun_name', 'doro_address', 'lat', 'lot']]
starbucks_df['lat'] = starbucks_df['lat'].astype(float)
starbucks_df['lot'] = starbucks_df['lot'].astype(float)

foliumMap = folium.Map(location=[starbucks_df['lat'].mean(), starbucks_df['lot'].mean()], zoom_start=12)
for _, star in starbucks_df.iterrows():
    popup = folium.Popup(f'{star["s_name"]} - {star["doro_address"]}', max_width=300)
    folium.Marker(location=[star['lat'], star['lot']], popup=popup).add_to(foliumMap)
foliumMap

## 제주도

In [26]:
targetSite = 'https://www.starbucks.co.kr/store/getStore.do?r=P4K5ZAPLE6'
response = requests.post(url=targetSite, data={
    'ins_lat': 37.365,
    'ins_lng': 127.0639,
    'p_sido_cd': '16',
    'p_gugun_cd': '',
    'in_biz_cd': '',
    'iend': 2300,
    'set_date': ''
})
storeList = response.json()

starbucks_df = json_normalize(storeList, 'list')
starbucks_df = starbucks_df[['s_name', 'sido_code', 'sido_name', 'gugun_code', 'gugun_name', 'doro_address', 'lat', 'lot']]
starbucks_df['lat'] = starbucks_df['lat'].astype(float)
starbucks_df['lot'] = starbucks_df['lot'].astype(float)

foliumMap = folium.Map(location=[starbucks_df['lat'].mean(), starbucks_df['lot'].mean()], zoom_start=12)
for _, star in starbucks_df.iterrows():
    popup = folium.Popup(f'{star["s_name"]} - {star["doro_address"]}', max_width=300)
    folium.Marker(location=[star['lat'], star['lot']], popup=popup).add_to(foliumMap)
foliumMap

전국 스타벅스 지점 시각화

In [29]:
targetSite = 'https://www.starbucks.co.kr/store/getStore.do?r=P4K5ZAPLE6'
response = requests.post(url=targetSite, data={
    'ins_lat': 37.365,
    'ins_lng': 127.0639,
    'p_sido_cd': '',
    'p_gugun_cd': '',
    'in_biz_cd': '',
    'iend': 2300,
    'set_date': ''
})
storeList = response.json()

starbucks_df = json_normalize(storeList, 'list')
starbucks_df = starbucks_df[['s_name', 'sido_code', 'sido_name', 'gugun_code', 'gugun_name', 'doro_address', 'lat', 'lot']]
starbucks_df['lat'] = starbucks_df['lat'].astype(float)
starbucks_df['lot'] = starbucks_df['lot'].astype(float)

foliumMap = folium.Map(location=[starbucks_df['lat'].mean(), starbucks_df['lot'].mean()], zoom_start=8)
for _, star in starbucks_df.iterrows():
    popup = folium.Popup(f'{star["s_name"]} - {star["doro_address"]}', max_width=300)
    folium.Marker(location=[star['lat'], star['lot']], popup=popup).add_to(foliumMap)
foliumMap

## 판다스 데이터프레임을 csv 파일로 저장하기

In [30]:
starbucks_df.to_csv('./data/starbucks.csv', index=False, encoding='utf-8-sig')

## csv 파일을 판다스에서 읽어서 데이터프레임으로 만들기

In [31]:
df = pd.read_csv('./data/starbucks.csv')
df

,s_name,sido_code,sido_name,gugun_code,gugun_name,doro_address,lat,lot
0,원주혁신도시,9,강원도,916,원주시,강원특별자치도 원주시 혁신로 61 (반곡동),37.323266,127.980932
1,엔터식스,1,서울,123,성동구,서울특별시 성동구 왕십리광장로 17 (행당동),37.561863,127.037852
2,서울타워,1,서울,124,용산구,서울특별시 용산구 남산공원길 105 (용산동2가) YTN서울타워 1층,37.551102,126.988317
3,한티역,1,서울,101,강남구,"서울특별시 강남구 도곡로 408, 디마크빌딩 101호 (대치동)",37.496801,127.055055
4,부산대역,5,부산,501,금정구,"부산광역시 금정구 장전로12번길 64, 1~4층 (장전동)",35.230522,129.088694
...,...,...,...,...,...,...,...,...
2207,호계지식산업센터,8,경기,819,안양시 동안구,경기도 안양시 동안구 엘에스로 122 (호계동) 118~123호,37.372519,126.948725
2208,안동이마트,11,경북,1123,안동시,경상북도 안동시 옥동1길 2 (옥동),36.558505,128.699308
2209,월계역,1,서울,107,노원구,"서울특별시 노원구 초안산로2라길26 월계동(104,105,106호)",37.629500,127.057400
2210,광교갤러리아9F,8,경기,839,수원시 영통구,"경기도 수원시 영통구 광교중앙로 124, 갤러리아 광교 9층 (하동)",37.285343,127.057235
